In [0]:
# Gold layer — food inspection star schema

Builds the dimensional model from the silver snapshot `foodguard_ai.silver.food_inspections` (full recompute, overwrite each run):

* **`dim_facility`** — one row per facility (identified by `license_`); attributes from its most recent inspection record; `first_inspection_date` / `last_inspection_date` span all its inspections; `facility_key` is a stable bigint surrogate (`xxhash64(license_)`)
* **`dim_date`** — one row per inspection date; int `date_key` = `yyyyMMdd`; year/quarter/month names, weekend flag
* **`dim_violation`** — one row per cited violation code; tier is the highest severity observed for that code in the citation text (`PRIORITY` → critical, `PRIORITY FOUNDATION` → serious, otherwise minor/Core)
* **`fact_inspection`** — one row per inspection: `is_reinspection` / `is_complaint` / `is_pass` / `is_fail` flags, risk level + label (dataset quirk `risk` = "All" → nulls), violation counts by tier, and prior-inspection columns (previous event per facility: `prior_inspection_date`, `prior_result`, `days_since_prior`)
* **`fact_violation`** — one row per cited violation, parsed from the free-text `violations` column; `is_repeat_violation` = the same code was cited in the facility's immediately prior inspection

Grain and FKs: `fact_inspection` joins to `dim_facility` on `facility_key` and `dim_date` on `date_key`; `fact_violation` joins to `dim_violation` on `violation_code` and to `fact_inspection` on `inspection_id`.

In [0]:
# Configure: silver source and the five gold star-schema tables.
SILVER_TABLE = "foodguard_ai.silver.food_inspections"
GOLD = "foodguard_ai.gold"
DIM_FACILITY = f"{GOLD}.dim_facility"
DIM_DATE = f"{GOLD}.dim_date"
DIM_VIOLATION = f"{GOLD}.dim_violation"
FACT_INSPECTION = f"{GOLD}.fact_inspection"
FACT_VIOLATION = f"{GOLD}.fact_violation"

print("Gold tables:")
for t in (DIM_FACILITY, DIM_DATE, DIM_VIOLATION, FACT_INSPECTION, FACT_VIOLATION):
    print(" ", t)

In [0]:
# Read silver and parse the free-text `violations` column into one row per cited
# violation. Format: "NN. TITLE - Comments: TEXT | NN. TITLE - ...".
# Tier comes from the citation written into the comments:
#   "PRIORITY FOUNDATION" -> serious, "PRIORITY" (unqualified) -> critical, else minor.
# is_repeat_violation: the same code was cited in the facility's immediately prior
# inspection event (previous inspection_id per license, ordered by date then id).
import pyspark.sql.functions as F
from pyspark.sql.window import Window

silver_df = spark.table(SILVER_TABLE)
print(f"Silver rows: {silver_df.count()}")

items_all = (
    silver_df
    .where(F.col("violations").isNotNull())
    .select("*", F.posexplode(F.split(F.col("violations"), r"\s*\|\s*")).alias("pos", "item"))
    .where(F.trim(F.col("item")) != "")
)

items_df = (
    items_all
    .withColumn("violation_code", F.regexp_extract("item", r"^\s*(\d+)\.", 1).cast("int"))
    .where(F.col("violation_code").isNotNull())
    .withColumn("violation_seq", F.col("pos") + 1)
    .withColumn("_after_code", F.regexp_extract("item", r"^\s*\d+\.\s*(.*)$", 1))
    .withColumn(
        "violation_description",
        F.nullif(F.trim(F.regexp_extract("_after_code", r"^(.*?)\s*-\s*Comments:", 1)), F.lit("")),
    )
    .withColumn(
        "violation_comments",
        F.nullif(F.trim(F.regexp_extract("item", r"Comments:\s*(.*)$", 1)), F.lit("")),
    )
    .withColumn(
        "violation_tier",
        F.when(F.lower(F.col("violation_comments")).contains("priority foundation"), F.lit("serious"))
        .when(F.lower(F.col("violation_comments")).contains("priority"), F.lit("critical"))
        .otherwise(F.lit("minor")),
    )
    .drop("pos", "item", "_after_code")
)
skipped = items_all.count() - items_df.count()
print(f"Parsed violations: {items_df.count()} rows | skipped (no leading code): {skipped}")

# Repeat-violation flags: map each (license, event) to its previous event, then flag
# violations whose code appears in that previous event's citations.
event_seq = (
    silver_df
    .select("license_", "inspection_id", F.to_date("inspection_date").alias("inspection_date"))
    .withColumn("_prev_id", F.lag("inspection_id").over(Window.partitionBy("license_").orderBy("inspection_date", "inspection_id")))
    .where(F.col("_prev_id").isNotNull())
)
repeat_codes = (
    items_df.select("license_", "inspection_id", "violation_code").distinct()
    .withColumnRenamed("inspection_id", "_prev_id")
    .join(event_seq.select("license_", "_prev_id", "inspection_id"), ["license_", "_prev_id"], "inner")
    .select("license_", "inspection_id", "violation_code")
    .distinct()
    .withColumn("_rep", F.lit(True))
)
items_df = (
    items_df
    .join(repeat_codes, ["license_", "inspection_id", "violation_code"], "left")
    .withColumn("is_repeat_violation", F.col("_rep").isNotNull())
    .drop("_rep")
)
print(f"Repeat violations: {items_df.where(F.col('is_repeat_violation')).count()} rows")
display(items_df.limit(5))

In [0]:
# DIM_DATE: one row per distinct inspection date; int YYYYMMDD surrogate key.
date_df = (
    silver_df
    .select(F.to_date("inspection_date").alias("date"))
    .where(F.col("date").isNotNull())
    .distinct()
)

dim_date_df = date_df.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),
    F.year("date").alias("year"),
    F.quarter("date").alias("quarter"),
    F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.dayofweek("date").isin(1, 7).alias("is_weekend"),  # 1 = Sunday, 7 = Saturday
)

dim_date_df.write.mode("overwrite").saveAsTable(DIM_DATE)
print(f"{DIM_DATE}: {dim_date_df.count()} rows")
display(dim_date_df.orderBy("date_key").limit(5))

In [0]:
# DIM_VIOLATION: one row per cited violation code. Description = the code's most
# common title; tier = the highest severity ever observed for the code (a code that
# was ever cited as Priority is critical), since severity is written per citation.
tier_rank = (
    F.when(F.col("violation_tier") == "critical", 3)
    .when(F.col("violation_tier") == "serious", 2)
    .otherwise(1)
)

dim_violation_df = (
    items_df
    .groupBy("violation_code")
    .agg(
        F.mode("violation_description").alias("violation_description"),
        F.max(tier_rank).alias("_tier_rank"),
    )
    .withColumn(
        "violation_tier",
        F.when(F.col("_tier_rank") == 3, F.lit("critical"))
        .when(F.col("_tier_rank") == 2, F.lit("serious"))
        .otherwise(F.lit("minor")),
    )
    .withColumn("is_critical", F.col("violation_tier") == "critical")
    .drop("_tier_rank")
)

dim_violation_df.write.mode("overwrite").saveAsTable(DIM_VIOLATION)
print(f"{DIM_VIOLATION}: {dim_violation_df.count()} rows")
display(dim_violation_df.orderBy(F.desc("violation_code")).limit(10))

In [0]:
# DIM_FACILITY: one row per facility (license). Attributes come from the facility's
# most recent inspection record (by date, then :updated_at); first/last inspection
# dates span all its inspections. facility_key = stable bigint surrogate xxhash64(license).
w_fac = Window.partitionBy("license_").orderBy(
    F.col("inspection_date").desc(), F.col(":updated_at").desc()
)
fac_latest = (
    silver_df
    .withColumn("_rn", F.row_number().over(w_fac))
    .where(F.col("_rn") == 1)
    .select(
        "license_", "dba_name", "aka_name", "facility_type",
        "address", "city", "state", "zip", "latitude", "longitude",
    )
)

dim_facility_df = (
    silver_df
    .groupBy("license_")
    .agg(
        F.min(F.to_date("inspection_date")).alias("first_inspection_date"),
        F.max(F.to_date("inspection_date")).alias("last_inspection_date"),
    )
    .join(fac_latest, "license_")
    .select(
        F.xxhash64(F.col("license_").cast("string")).alias("facility_key"),
        F.col("license_").alias("license_no"),
        "dba_name", "aka_name", "facility_type", "address", "city", "state",
        F.col("zip").cast("string").alias("zip"),
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
        "first_inspection_date", "last_inspection_date",
    )
)

dim_facility_df.write.mode("overwrite").saveAsTable(DIM_FACILITY)
print(f"{DIM_FACILITY}: {dim_facility_df.count()} rows (distinct licenses)")
display(dim_facility_df.limit(5))

In [0]:
# FACT_INSPECTION: one row per inspection (inspection_id is unique in silver).
# * risk "All" (dataset quirk, 29 rows) keeps null level/label — it is not a real risk tier.
# * violation counts are joined from the parsed citations (inspections with no cited
#   violations get zeros).
# * prior-inspection columns use the facility's previous event (ordered by date, then id).
w_lic = Window.partitionBy("license_").orderBy("inspection_date", "inspection_id")
prior_df = (
    silver_df
    .withColumn("_prior_ts", F.lag("inspection_date").over(w_lic))
    .withColumn("_prior_result", F.lag("results").over(w_lic))
)

counts_df = items_df.groupBy("inspection_id").agg(
    F.count(F.lit(1)).alias("violation_count"),
    F.sum((F.col("violation_tier") == "critical").cast("int")).alias("critical_count"),
    F.sum((F.col("violation_tier") == "serious").cast("int")).alias("serious_count"),
    F.sum((F.col("violation_tier") == "minor").cast("int")).alias("minor_count"),
    F.sum(F.col("is_repeat_violation").cast("int")).alias("repeat_violation_count"),
)

fact_inspection_df = (
    prior_df
    .join(counts_df, "inspection_id", "left")
    .select(
        F.col("inspection_id"),
        F.xxhash64(F.col("license_").cast("string")).alias("facility_key"),
        F.date_format("inspection_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("inspection_type"),
        F.lower(F.col("inspection_type")).contains("re-inspection").alias("is_reinspection"),
        F.lower(F.col("inspection_type")).contains("complaint").alias("is_complaint"),
        F.nullif(F.regexp_extract("risk", r"^Risk (\d)", 1), F.lit("")).cast("int").alias("risk_level"),
        F.nullif(F.regexp_extract("risk", r"\(([^)]*)\)", 1), F.lit("")).alias("risk_label"),
        F.col("results").alias("result"),
        F.col("results").isin("Pass", "Pass w/ Conditions").alias("is_pass"),
        (F.col("results") == "Fail").alias("is_fail"),
        F.coalesce(F.col("violation_count"), F.lit(0)).cast("int").alias("violation_count"),
        F.coalesce(F.col("critical_count"), F.lit(0)).cast("int").alias("critical_count"),
        F.coalesce(F.col("serious_count"), F.lit(0)).cast("int").alias("serious_count"),
        F.coalesce(F.col("minor_count"), F.lit(0)).cast("int").alias("minor_count"),
        F.coalesce(F.col("repeat_violation_count"), F.lit(0)).cast("int").alias("repeat_violation_count"),
        F.to_date("_prior_ts").alias("prior_inspection_date"),
        F.col("_prior_result").alias("prior_result"),
        F.datediff(F.to_date("inspection_date"), F.to_date("_prior_ts")).alias("days_since_prior"),
    )
)

fact_inspection_df.write.mode("overwrite").saveAsTable(FACT_INSPECTION)
print(f"{FACT_INSPECTION}: {fact_inspection_df.count()} rows")
display(fact_inspection_df.limit(5))

In [0]:
# FACT_VIOLATION: one row per cited violation. violation_id = deterministic sha2 of
# (inspection_id, violation_code, violation_seq) — the same code can be cited more
# than once in a single inspection, so the sequence number disambiguates.
fact_violation_df = items_df.select(
    F.sha2(
        F.concat_ws(
            "||",
            F.col("inspection_id").cast("string"),
            F.col("violation_code").cast("string"),
            F.col("violation_seq").cast("string"),
        ),
        256,
    ).alias("violation_id"),
    "inspection_id",
    F.xxhash64(F.col("license_").cast("string")).alias("facility_key"),
    "violation_code",
    "violation_seq",
    F.col("violation_comments").alias("violation_comments"),
    "is_repeat_violation",
    F.to_date("inspection_date").alias("inspection_date"),
)

fact_violation_df.write.mode("overwrite").saveAsTable(FACT_VIOLATION)
print(f"{FACT_VIOLATION}: {fact_violation_df.count()} rows")
display(fact_violation_df.limit(5))

In [0]:
# Verify the star schema: PK uniqueness, FK integrity, and fact-count consistency.
fact_i = spark.table(FACT_INSPECTION)
dim_f = spark.table(DIM_FACILITY)
dim_d = spark.table(DIM_DATE)
dim_v = spark.table(DIM_VIOLATION)
fact_v = spark.table(FACT_VIOLATION)

print("PK uniqueness:")
pk_ok = True
for table, df, key in [
    (DIM_FACILITY, dim_f, "facility_key"),
    (DIM_DATE, dim_d, "date_key"),
    (DIM_VIOLATION, dim_v, "violation_code"),
    (FACT_INSPECTION, fact_i, "inspection_id"),
    (FACT_VIOLATION, fact_v, "violation_id"),
]:
    n = df.count()
    ok = df.select(key).distinct().count() == n
    pk_ok = pk_ok and ok
    print(f"  {table}: {ok} ({n} rows)")

fk_facility = fact_i.join(dim_f, "facility_key", "left_anti").count() == 0
fk_date = fact_i.join(dim_d, "date_key", "left_anti").count() == 0
fk_fv_facility = fact_v.join(dim_f, "facility_key", "left_anti").count() == 0
fk_fv_code = fact_v.join(dim_v, "violation_code", "left_anti").count() == 0
fk_fv_inspection = fact_v.join(fact_i, "inspection_id", "left_anti").count() == 0
print(f"FK integrity — fact_inspection → dim_facility: {fk_facility} | dim_date: {fk_date}")
print(f"FK integrity — fact_violation → dim_facility: {fk_fv_facility} | dim_violation: {fk_fv_code} | fact_inspection: {fk_fv_inspection}")

sums = fact_i.agg(
    F.sum("violation_count"), F.sum("critical_count"),
    F.sum("serious_count"), F.sum("minor_count"),
).collect()[0]
viol_total = fact_v.count()
tier_total = (sums["sum(critical_count)"] or 0) + (sums["sum(serious_count)"] or 0) + (sums["sum(minor_count)"] or 0)
print(f"Consistency — sum(violation_count)={sums['sum(violation_count)']} vs fact_violation rows={viol_total} | tier counts sum={tier_total}")

assert pk_ok, "PK duplication detected!"
assert fk_facility and fk_date and fk_fv_facility and fk_fv_code and fk_fv_inspection, "FK integrity violated!"
assert sums["sum(violation_count)"] == viol_total, "violation_count does not match fact_violation!"
assert tier_total == sums["sum(violation_count)"], "tier counts do not add up!"
print("All gold-layer checks passed.")